# StochastiQ — Notebook 01: Data Pull

**Project:** StochastiQ — Multi-model portfolio optimization and derivatives strategy framework  
**Course:** MGT 6081 Derivative Securities, Georgia Institute of Technology  
**Author:** Anay Abhijit Joshi

---

## Objectives

1. Acquire 5 years of daily adjusted closing prices for the 7-asset universe via Yahoo Finance.
2. Perform data quality checks and cleaning.
3. Compute simple and log returns.
4. Generate annualized summary statistics.
5. Visualize prices, returns, rolling volatility, and the correlation structure.
6. Persist cleaned datasets to `data/processed/` for downstream notebooks.

## Universe

Seven assets, deliberately heterogeneous to exhibit a range of stochastic behaviors:

| Ticker | Asset | Role |
|--------|-------|------|
| AAPL | Apple Inc. | Large-cap technology |
| MSFT | Microsoft Corp. | Large-cap technology |
| JPM | JPMorgan Chase | Financials, rate-sensitive |
| JNJ | Johnson & Johnson | Defensive healthcare |
| XOM | Exxon Mobil | Energy, commodity-linked |
| SPY | SPDR S&P 500 ETF | Broad market benchmark |
| GLD | SPDR Gold Trust | Crisis hedge |

This selection ensures the portfolio captures meaningful correlation diversity, which is essential for the multi-model robustness analysis later in the project.

## 1. Setup

In [ ]:
# Make src/ importable
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loaders import (
    DEFAULT_UNIVERSE,
    ASSET_DESCRIPTIONS,
    fetch_prices,
    data_quality_report,
    clean_prices,
    compute_returns,
    annualized_summary,
    save_dataset,
)

# Plot styling
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (12, 5)

# Output paths
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Universe ({len(DEFAULT_UNIVERSE)} assets): {DEFAULT_UNIVERSE}")

## 2. Fetch Historical Prices

We pull 5+ years of daily adjusted closing prices, starting from January 1, 2020. Using `auto_adjust=True` ensures prices are adjusted for splits and dividends, which is critical for accurate return computation.

In [ ]:
START_DATE = "2020-01-01"
END_DATE = None  # None means today

prices_raw = fetch_prices(
    tickers=DEFAULT_UNIVERSE,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
)

print(f"Shape: {prices_raw.shape}")
print(f"Date range: {prices_raw.index.min().date()} to {prices_raw.index.max().date()}")
prices_raw.head()

## 3. Data Quality Check

Before using the data, we audit it for missing values, zero/negative prices (which would indicate data errors), and verify date alignment across tickers.

In [ ]:
quality = data_quality_report(prices_raw)
quality

In [ ]:
# Clean prices: forward-fill isolated missing days, drop any remaining gaps
prices = clean_prices(prices_raw)

print(f"Raw shape:     {prices_raw.shape}")
print(f"Cleaned shape: {prices.shape}")
print(f"Rows dropped:  {len(prices_raw) - len(prices)}")

assert prices.isna().sum().sum() == 0, "Cleaned prices still contain NaN values"
assert (prices > 0).all().all(), "Cleaned prices contain zero or negative values"
print("\n[OK] Cleaned prices are complete and positive across all assets.")

## 4. Compute Returns

We compute both simple returns ($r_t = P_t / P_{t-1} - 1$) and log returns ($r_t = \ln(P_t / P_{t-1})$).

- **Log returns** are used for model calibration because they are time-additive and approximately normal under GBM.
- **Simple returns** are used for portfolio aggregation since portfolio simple returns are linear combinations of asset simple returns.

In [ ]:
returns = compute_returns(prices)
simple_returns = returns["simple"]
log_returns = returns["log"]

print(f"Simple returns shape: {simple_returns.shape}")
print(f"Log returns shape:    {log_returns.shape}")
log_returns.head()

## 5. Annualized Summary Statistics

Annualization assumes 252 trading days per year. The Sharpe ratio uses a 4% risk-free rate, reflecting the current short-term Treasury environment.

In [ ]:
summary = annualized_summary(log_returns, trading_days=252, risk_free_rate=0.04)
summary.style.format({
    "annual_return":     "{:.2%}",
    "annual_volatility": "{:.2%}",
    "sharpe_ratio":      "{:.3f}",
    "skewness":          "{:.3f}",
    "excess_kurtosis":   "{:.3f}",
}).background_gradient(subset=["sharpe_ratio"], cmap="RdYlGn")

**Interpretation hints:**
- Excess kurtosis significantly above 0 indicates fat tails — relevant for justifying Merton (jump-diffusion) and Heston (stochastic volatility) over plain GBM.
- Negative skewness is typical for equities (crash risk) and supports the use of jump models.
- These observations will be revisited during model selection in Notebook 03.

## 6. Visualization

### 6.1 Price Evolution (Normalized to 100)

In [ ]:
normalized = prices.divide(prices.iloc[0]).multiply(100)

fig, ax = plt.subplots(figsize=(13, 6))
for ticker in normalized.columns:
    ax.plot(normalized.index, normalized[ticker], label=ticker, linewidth=1.5)
ax.set_title("Normalized Price Evolution (Base = 100)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Normalized Price")
ax.legend(loc="best", ncol=4)
ax.axhline(100, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_normalized_prices.png", bbox_inches="tight")
plt.show()

### 6.2 Return Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=False)
axes = axes.flatten()

for i, ticker in enumerate(log_returns.columns):
    ax = axes[i]
    data = log_returns[ticker]
    ax.hist(data, bins=60, density=True, alpha=0.6, edgecolor="black", linewidth=0.3)
    
    # Overlay normal distribution for comparison
    x = np.linspace(data.min(), data.max(), 200)
    mu, sigma = data.mean(), data.std()
    normal_pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    ax.plot(x, normal_pdf, "r-", linewidth=1.8, label="Normal fit")
    
    ax.set_title(f"{ticker}\n(skew={data.skew():.2f}, kurt={data.kurtosis():.2f})", fontsize=10)
    ax.legend(fontsize=8)

# Hide the unused 8th subplot
axes[7].set_visible(False)

fig.suptitle("Daily Log Return Distributions vs. Normal Fit", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_return_distributions.png", bbox_inches="tight")
plt.show()

Visible departures from the red normal curve in the tails justify the use of non-Gaussian models (Merton, Heston) downstream.

### 6.3 Rolling 30-Day Annualized Volatility

In [ ]:
rolling_vol = log_returns.rolling(window=30).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(13, 6))
for ticker in rolling_vol.columns:
    ax.plot(rolling_vol.index, rolling_vol[ticker], label=ticker, linewidth=1.2, alpha=0.85)
ax.set_title("30-Day Rolling Annualized Volatility", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized Volatility")
ax.legend(loc="best", ncol=4)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_rolling_volatility.png", bbox_inches="tight")
plt.show()

Volatility clustering — the tendency for high-volatility days to cluster together — is clearly visible. This empirical fact directly motivates the Heston stochastic volatility model in Notebook 03.

### 6.4 Correlation Matrix

In [ ]:
corr_matrix = log_returns.corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Correlation"},
    ax=ax,
)
ax.set_title("Pairwise Daily Log-Return Correlations", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_correlation_matrix.png", bbox_inches="tight")
plt.show()

Strong positive correlations within equity assets, with GLD providing meaningful diversification (low correlation to the equity block). This correlation structure justifies the asset selection.

## 7. Save Cleaned Datasets

We save outputs as Parquet files (efficient binary format, fast IO, smaller on disk than CSV).

In [ ]:
saved_files = []
saved_files.append(save_dataset(prices, PROCESSED_DIR / "prices", fmt="parquet"))
saved_files.append(save_dataset(simple_returns, PROCESSED_DIR / "simple_returns", fmt="parquet"))
saved_files.append(save_dataset(log_returns, PROCESSED_DIR / "log_returns", fmt="parquet"))
saved_files.append(save_dataset(summary, PROCESSED_DIR / "summary_stats", fmt="parquet"))

print("Saved files:")
for f in saved_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.relative_to(PROJECT_ROOT)} ({size_kb:.1f} KB)")

## 8. Summary

| Item | Value |
|------|-------|
| Universe | 7 assets |
| Date range | 2020-01-01 to today |
| Trading days | (see shape above) |
| Data quality | Verified: no missing values, no zero/negative prices |
| Datasets saved | `prices`, `simple_returns`, `log_returns`, `summary_stats` |
| Figures saved | `reports/figures/01_*.png` |

**Key empirical observations** (to inform model selection in Notebook 03):
- Returns exhibit fat tails (excess kurtosis > 0) → justifies Merton jump-diffusion and Heston.
- Negative skewness is present in equity assets → justifies asymmetric jump distributions.
- Volatility clusters in time → directly motivates Heston's stochastic variance process.
- GLD provides genuine diversification (low correlation to equities) → useful for portfolio construction.

**Next:** Notebook 02 — Markowitz mean-variance optimization with alternative objective functions (Sortino, CVaR, Risk Parity).